In [12]:
import pandas as pd
import numpy as numpy

# loading datasets
df_flights = pd.read_csv("../data/processed/cleaned_flights.csv")
df_hotels = pd.read_csv("../data/processed/cleaned_hotels.csv")
df_trends = pd.read_csv("../data/processed/cleaned_trends.csv")

# turn dates into date objects
df_flights["date_collected"] = pd.to_datetime(df_flights["datetime_collected"]).dt.date
df_hotels["date_collected"] = pd.to_datetime(df_hotels["datetime_collected"]).dt.date
# (trends is a special case since its a UNIX timestamp)
df_trends["date"] = pd.to_datetime(df_trends["timestamp"], unit='s', errors='coerce')
if df_trends["date"].isnull().all():
    df_trends["date"] = pd.to_datetime(df_trends["timestamp"]).dt.date
else:
    df_trends["date"] = df_trends["date"].dt._delegate_method

# ensure that the values are numeric
df_flights["price"] = pd.to_numeric(df_flights["price"], errors="coerce")
df_hotels["price_php"] = pd.to_numeric(df_hotels["price_php"], errors="coerce")
df_trends["value"] = pd.to_numeric(df_trends["value"], errors="coerce")

In [13]:
# aggregate data by day of collection

daily_flights = df_flights.groupby("date_collected").agg(
    flight_min_price=("price", "min"),
    flight_max_price=("price", "max"),
    flight_median_price=("price", "median"),
    flight_mean_price=("price", "mean"),
    flight_quote_count=("price", "count")
).reset_index()

daily_hotels = df_hotels.groupby("date_collected").agg(
    hotel_min_price=("price", "min"),
    hotel_max_price=("price", "max"),
    hotel_median_price=("price", "median"),
    hotel_mean_price=("price", "mean"),
    hotel_quote_count=("price", "count")
).reset_index()

daily_trends = df_trends.groupby("date").agg(
    trend_search_score=("value", "max")
).reset_index().rename(columns={"date":"date_collected"})

KeyError: "Label(s) ['price'] do not exist"

In [ ]:
# make the master_df and sort by date collected (which should be a datetime obj)

master_df = daily_trends.merge(daily_flights, on="date_collected", how="left")
master_df = master_df.merge(daily_hotels, on="date_collected", how="left")

master_df = master_df.sort_values("date_collected").reset_index(drop=True)

In [ ]:
# feature engineering and derived metrics, apparently

In [ ]:
# trend velocity
master_df["trend_velociy_7d"] = master_df["trend_search_score"].pct_change(periods=7)

In [ ]:
# lagg time to surge
master_df["trend_score_lag_1d"] = master_df["trend_search_score"].shift(1)
master_df["trend_score_lag_2d"] = master_df["trend_search_score"].shift(2)
master_df["trend_score_lag_3d"] = master_df["trend_search_score"].shift(3)

In [ ]:
# price velocity
master_df["flight_median_change_id"] = master_df["flight_median_price"].diff(1)
master_df["hotel_median_change_id"] = master_df["hotel_median_price"].diff(1)

In [ ]:
# lead time
festival_start = pd.to_datetime("2026-10-09").date()
master_df["days_to_festival"] = master_df["date_collected"].apply(lambda d: (festival_start - d).days if pd.notnull(d) else np.nan)

In [ ]:
# missing values handling and saving

output_path = "../data/processed/masskara/analysis/master.csv"
master_df.to_csv(output_path, index=False)